# Webscrape site URLS

In [1]:
from bs4 import BeautifulSoup
import pandas as pd
import requests


URL = "https://whc.unesco.org/en/list/"
page = requests.get(URL)

with open("unesco_list.html", "w", encoding="utf-8") as file:
    file.write(page.text)

In [2]:
df = pd.DataFrame(columns=["name", "type", "link"])
with open("unesco_list.html", "r", encoding="utf-8") as file:
    html_content = file.read()

soup = BeautifulSoup(html_content, "html.parser")
tables = soup.find_all("div", class_="list_site")

data_list = []

for table in tables:
    for li in table.find_all("li"):
        classes = li.get("class", [])
        site_type = classes[0] if classes else None

        a = li.find("a")
        if a:
            href = "https://whc.unesco.org" + a.get("href")
            name = a.get_text(strip=True)

            data_list.append({
                "name": name, 
                "site_type": site_type,
                "link": href
            })

df = pd.DataFrame(data_list)


In [3]:
df["unique_id"] = df["link"].apply(lambda x: x.split('/')[-1] if isinstance(x, str) else None)

In [4]:
df.to_csv("unesco_list.csv", index=False)

# Add columns for gallery URL and picture URL

In [8]:
import pandas as pd

df = pd.read_csv("unesco_list.csv")

# build gallery URLs safely
df["gallery_url"] = df["link"].astype(str).str.rstrip("/") + "/gallery"
df["picture_url"] = None
print(df[df["gallery_url"] == "https://whc.unesco.org/en/list/1361/gallery"])

                                     name site_type  \
1040  Historic Jeddah, the Gate to Makkah  cultural   

                                     link  unique_id  \
1040  https://whc.unesco.org/en/list/1361       1361   

                                      gallery_url picture_url  
1040  https://whc.unesco.org/en/list/1361/gallery        None  


In [9]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import time 
import random
import sys

URL = "https://whc.unesco.org/en/list/102/gallery"

opts = Options()
opts.add_argument("--headless=new")
opts.add_argument("--no-sandbox")
opts.add_argument("--disable-dev-shm-usage")
opts.add_argument("--window-size=1400,900")
opts.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=opts)

def get_picture(URL, driver):
    if not isinstance(URL, str) or pd.isna(URL):
        return ""
    try:
        time.sleep(random.uniform(1, 3))
        driver.get(URL)
        soup = BeautifulSoup(driver.page_source, "html.parser")
        og = soup.select_one('meta[property="og:image"]')
        
        result = og["content"] if og else ""
        print(f"\rFetching: {URL} --> Picture: {result}               ", end="", flush=True)
        return result
    except Exception as e:
        return ""

mask = df["picture_url"].isna() & df["gallery_url"].notna()

df.loc[mask, "picture_url"] = df.loc[mask, "gallery_url"].apply(
    lambda x: get_picture(x, driver)
)

driver.quit()
print("finished scraping")

Fetching: https://whc.unesco.org/en/list/306/gallery --> Picture: https://whc.unesco.org/uploads/thumbs/site_0306_0001-1200-630-20151105163117.jpgg

In [11]:
df["picture_url"].head(5)
df.head(5)

,name,site_type,link,unique_id,gallery_url,picture_url
0,Minaret and Archaeological Remains of Jam,cultural_danger,https://whc.unesco.org/en/list/211,211,https://whc.unesco.org/en/list/211/gallery,https://whc.unesco.org/uploads/thumbs/site_021...
1,Cultural Landscape and Archaeological Remains ...,cultural_danger,https://whc.unesco.org/en/list/208,208,https://whc.unesco.org/en/list/208/gallery,https://whc.unesco.org/uploads/thumbs/site_020...
2,Natural and Cultural Heritage of the Ohrid region,mixed,https://whc.unesco.org/en/list/99,99,https://whc.unesco.org/en/list/99/gallery,https://whc.unesco.org/uploads/thumbs/site_009...
3,Butrint,cultural,https://whc.unesco.org/en/list/570,570,https://whc.unesco.org/en/list/570/gallery,https://whc.unesco.org/uploads/thumbs/site_057...
4,Historic Centres of Berat and Gjirokastra,cultural,https://whc.unesco.org/en/list/569,569,https://whc.unesco.org/en/list/569/gallery,https://whc.unesco.org/uploads/thumbs/site_056...


In [12]:
df.to_csv("unesco_list.csv", index=False)

# Construction dates


In [63]:
# Regex date extraction was removed.
# Generate sites.geojson first, then run the cached LLM enrichment script from the repo root:
# py -B convert/extract_construction_history.py


In [64]:
# The LLM script writes properties.construction_history and updates properties.date for the frontend timeline.


(1501, 1600, 'AD')

# Convert to .json

In [73]:
import pandas as pd
import json

df = pd.read_csv("./unesco_sites.csv")
df_urls = pd.read_csv("./unesco_list.csv")
df_combined = pd.merge(df, df_urls, left_on="id_no", right_on="unique_id", how="left")
df_combined = df_combined.astype(object).where(pd.notnull(df_combined), None)

df_combined["short_description_en"] = (
    df_combined["short_description_en"]
      .astype(str)
      .str.replace("<p>", "", regex=False)
      .str.replace("</p>", "", regex=False)
)


features = []
for _, r in df_combined.iterrows():
    if r["longitude"] is None or r["latitude"] is None:
        continue

    if isinstance(r["date"], (tuple, list)) and len(r["date"]) == 3:
        s_raw, e_raw, bc_ad = r["date"]
        start_year = int(s_raw) if s_raw is not None else None
        end_year = int(e_raw) if e_raw is not None else None
    else:
        start_year, end_year, bc_ad = (None, None, None)

    features.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [float(r["longitude"]), float(r["latitude"])]},
        "properties": {
            "name_en": r["name_en"],
            "short_description_en": r["short_description_en"],
            "states_name_en": r["states_name_en"],
            "id_no": r["id_no"],
            "heritage_category": r["category"],
            "danger": int(r["danger"]) if r["danger"] is not None else 0,
            "gallery_url": r["gallery_url"], 
            "picture_url": r["picture_url"],
            "date": {"start_date": start_year, "end_date": end_year, "BC_AD": bc_ad}
        }
    })

geojson = {"type": "FeatureCollection", "features": features}

with open("sites.geojson", "w", encoding="utf-8") as f:
    json.dump(geojson, f, ensure_ascii=False, allow_nan=False)